In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 

#libraries for model training and eval
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

#saves the model so we dont have to retrain again
import joblib

In [ ]:
df = pd.read_csv("../2_pose_extraction/stroke_csv/all_landmarks.csv")
print(df.shape)
print(df['label'].value_counts())
print(df.columns.tolist())

# Training Model on dataset gathered

## Seperating which body landmarks are kept

In [ ]:
#not keeping visability for now
feature_columns = [
    'RIGHT_WRIST_x', 'RIGHT_WRIST_y', 'RIGHT_WRIST_z',
    'RIGHT_ELBOW_x', 'RIGHT_ELBOW_y', 'RIGHT_ELBOW_z',
    'RIGHT_SHOULDER_x', 'RIGHT_SHOULDER_y', 'RIGHT_SHOULDER_z',
    'LEFT_SHOULDER_x', 'LEFT_SHOULDER_y', 'LEFT_SHOULDER_z',
    'LEFT_HIP_x', 'LEFT_HIP_y', 'LEFT_HIP_z',
    'RIGHT_HIP_x', 'RIGHT_HIP_y', 'RIGHT_HIP_z',
    'RIGHT_ANKLE_x', 'RIGHT_ANKLE_y', 'RIGHT_ANKLE_z',
    'LEFT_ANKLE_x', 'LEFT_ANKLE_y', 'LEFT_ANKLE_z',
]

print(f"Keeping {len(feature_columns)} features")

In [ ]:
#create pd for the cols that we want to train with
X = df[feature_columns] # X is for independent variables
y = df['label'] # y is for dependent variables: what we trying to predict. lowercase for single col

# print(X.shape)
# print(y.shape)
# print(feature_columns)

## Splitting Data into 70% for training and 30% for testing

In [ ]:
#X and y train contain 70% of the cols, xy test contains the other 30%
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size= .3,   #sets testing to 30%, so 70% training
    random_state= 42    #sets a seed for reproducability
    )


## Training classifer using Random Forest

In [ ]:
clf = RandomForestClassifier(
    n_estimators= 100, #uses 100 decisions trees
    random_state= 42,   #sets seed
    n_jobs= -1      #uses all cores to train
)

clf.fit(X_train, y_train)
print("Finished training")

## Model Prediction and Error Rates

In [ ]:
y_pred = clf.predict(X_test)

print(f"Predictions made on {len(y_pred)} frames")
print(f"Sample predictions: {y_pred[:10]}")
print(f"Actual results: {y_test.values[:10]}")

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy:.3%}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)

#confusion matrix output
# [[ 305    0   27    0]    labled 305 bh correct, 27 as neutral
#  [   0  157   11    0]    157 fore, 11 wrong as neutral 
#  [  10    4 1229    1]        1229 neutral, 10 wrong as bh 4 as fh and 1 as serve
#  [   0    0    9  163]]

print(cm)
print(classification_report(y_test,y_pred))

#classification_report output: precision = how often right, recall = how many did i catch
#                   precision    recall  f1-score   support

#     Backhand       0.97      0.92      0.94       332        right- 97% , caught 92% 
#     Forehand       0.98      0.93      0.95       168
#      Neutral       0.96      0.99      0.98      1244
#        Serve       0.99      0.95      0.97       172

#     accuracy                           0.97      1916
#    macro avg       0.98      0.95      0.96      1916
# weighted avg       0.97      0.97      0.97      1916


In [ ]:
#Saves the ml model 

joblib.dump(clf, "stroke_classifer_model.pkl")
print("Saved as: stroke_classifer_model.pkl")

Saved as: stroke_classifer_model.pkl
